**Data Loading**

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from dateutil import parser
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report


In [ ]:
df1 = pd.read_csv('/content/Iteration3_SA_QLD.csv')
df2 = pd.read_csv('/content/Iteration3_VIC.csv')
df3= pd.read_csv('/content/Iteration3_Species_otherstates.csv')

In [ ]:
# df1 = df1.drop_duplicates(subset=['species_scientific_name','observation_date','latitude','longitude'])
# df2 = df2.drop_duplicates(subset=['species_scientific_name','observation_date','latitude','longitude'])

In [ ]:
df = pd.concat([df1, df2, df3], ignore_index=True)

In [ ]:
df.head(5)

,species_common_name,species_scientific_name,observation_date,month,state,latitude,longitude,observation_count,plant_common_name,plant_scientific_name,optimal_planting_months,growth_duration_weeks,flowering_period,sunlight_requirement,water_requirement,suitable_regions,is_endangered
0,Ostrich,Struthio camelus,9/11/2022,November,Queensland,-21.65029,138.11203,8,Kangaroo Paw,Anigozanthos spp.,Aug-Dec,12.0,Aug-Dec,Full Sun,Low,"QLD, SA",NaN
1,Ostrich,Struthio camelus,8/02/2022,February,Queensland,-13.09540,153.27852,6,Kangaroo Paw,Anigozanthos spp.,Aug-Dec,12.0,Aug-Dec,Full Sun,Low,"QLD, SA",NaN
2,Ostrich,Struthio camelus,9/12/2022,December,Queensland,-14.78864,146.63214,2,Bottlebrush,Callistemon spp.,Sep-Nov,10.0,Sep-Nov,Full Sun,Moderate,"QLD, SA",NaN
3,Ostrich,Struthio camelus,6/09/2022,September,Queensland,-11.38336,148.32343,13,Kangaroo Paw,Anigozanthos spp.,Aug-Dec,12.0,Aug-Dec,Full Sun,Low,"QLD, SA",NaN
4,Ostrich,Struthio camelus,1/01/2023,January,Queensland,-15.16180,144.77058,9,Bottlebrush,Callistemon spp.,Sep-Nov,10.0,Sep-Nov,Full Sun,Moderate,"QLD, SA",NaN


In [ ]:
# Check for null values in 'observation_date' column
null_count = df['observation_date'].isnull().sum()

if null_count > 0:
  print(f"There are {null_count} null values in the 'observation_date' column.")
else:
  print("There are no null values in the 'observation_date' column.")


There are 4605 null values in the 'observation_date' column.


In [ ]:
#check for missing values and duplicates

# Check for missing values
print(df.isnull().sum())
# Check for duplicates
print(df.duplicated().sum())


species_common_name          0
species_scientific_name    780
observation_date             0
month                        0
state                        0
latitude                     0
longitude                    0
observation_count            0
plant_common_name            0
plant_scientific_name        0
optimal_planting_months      0
growth_duration_weeks        0
flowering_period             0
sunlight_requirement         0
water_requirement            0
suitable_regions             0
dtype: int64
0


In [ ]:
df['state'].unique()

array(['Queensland', 'South Australia', 'Victoria'], dtype=object)

In [ ]:
#df = df.drop_duplicates(subset=['species_scientific_name','observation_date','latitude','longitude'])


In [ ]:
df.shape

(13815, 16)

**Data Cleaning**

In [ ]:
# 1. Flexible date parsing
df['observation_date'] = pd.to_datetime(
    df['observation_date'],
    dayfirst=True,
    infer_datetime_format=True,
    errors='coerce'
)

# 2. Check for any failures
bad = df['observation_date'].isna().sum()
print(f"Unparsed dates: {bad}")

# (Optional) If any, inspect a few:
print(df.loc[df['observation_date'].isna(), 'observation_date'].head())

# 3. Now derive year/month
df['year']       = df['observation_date'].dt.year
df['month_full'] = df['observation_date'].dt.month_name()
df['month']      = df['month_full'].str.slice(0,3)



Unparsed dates: 4605
9210   NaT
9211   NaT
9212   NaT
9213   NaT
9214   NaT
Name: observation_date, dtype: datetime64[ns]


<ipython-input-7-7f7b4b779daa>:2: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df['observation_date'] = pd.to_datetime(


In [ ]:
df.loc[[9210, 9211, 9212, 9213, 9214]]

,species_common_name,species_scientific_name,observation_date,month,state,latitude,longitude,observation_count,plant_common_name,plant_scientific_name,optimal_planting_months,growth_duration_weeks,flowering_period,sunlight_requirement,water_requirement,suitable_regions,year,month_full
9210,Ostrich,Struthio camelus,NaT,NaN,Victoria,-35.54394,147.75139,6,Bottlebrush,Callistemon spp.,Sep-Nov,10,Sep-Nov,Full Sun,Moderate,VIC,NaN,NaN
9211,Ostrich,Struthio camelus,NaT,NaN,Victoria,-38.17660,142.00771,13,Lilly Pilly,Syzygium spp.,Nov-Jan,16,Nov-Jan,Partial Sun,High,VIC,NaN,NaN
9212,Ostrich,Struthio camelus,NaT,NaN,Victoria,-35.95892,147.56197,4,Grevillea,Grevillea spp.,Jul-Oct,14,Jul-Oct,Full Sun,Low,VIC,NaN,NaN
9213,Ostrich,Struthio camelus,NaT,NaN,Victoria,-36.65431,149.62634,2,Wattle,Acacia spp.,Aug-Sep,8,Aug-Sep,Full Sun,Low,VIC,NaN,NaN
9214,Ostrich,Struthio camelus,NaT,NaN,Victoria,-36.77289,146.42699,8,Lilly Pilly,Syzygium spp.,Nov-Jan,16,Nov-Jan,Partial Sun,High,VIC,NaN,NaN


In [ ]:
bad_dates = df.loc[df['observation_date'].isna(), 'observation_date']
print(bad_dates.head(50).to_list())


[NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT]


In [ ]:
df_clean = df.dropna(subset=['observation_date'])
print(f"Remaining valid records: {len(df_clean)}")

Remaining valid records: 9210


In [ ]:
df_clean = df.dropna(subset=['observation_date']).copy()


In [ ]:
#Derive year/month fields
df_clean['year']       = df_clean['observation_date'].dt.year
df_clean['month_full'] = df_clean['observation_date'].dt.month_name()
df_clean['month']      = df_clean['month_full'].str.slice(0,3)




In [ ]:
#Infer state by latitude/longitude
conds = [
    df_clean['latitude'].astype(float).between(-29, -10) & df_clean['longitude'].astype(float).between(138, 154),
    df_clean['latitude'].astype(float).between(-38, -25) & df_clean['longitude'].astype(float).between(129, 141),
    df_clean['latitude'].astype(float).between(-39, -34) & df_clean['longitude'].astype(float).between(141, 150),
]
choices = ['Queensland','South Australia','Victoria']
df_clean['state'] = np.select(conds, choices, default=df_clean.get('state'))

In [ ]:
#Fix observation counts & ensure plant columns
df_clean['observation_count'] = pd.to_numeric(df_clean['observation_count'], errors='coerce').fillna(1).astype(int)

for col in ['plant_common_name','optimal_planting_months','growth_duration_weeks',
            'sunlight_requirement','water_requirement','suitable_regions']:
    if col not in df_clean.columns:
        df_clean[col] = pd.NA


In [ ]:
#Flag migratory species (species seen in >1 state)
visits = df_clean.groupby('species_scientific_name')['state'].nunique()
df_clean['is_migratory'] = df_clean['species_scientific_name'].isin(visits[visits>1].index)

In [ ]:
df_clean.to_csv('merged_birds_cleaned.csv', index=False)
print("Cleaned dataset saved as merged_birds_plants_cleaned.csv")

Cleaned dataset saved as merged_birds_plants_cleaned.csv


**Merging Datasets to get Endangered species**

In [2]:
vic_df = pd.read_csv('/content/Iteration3_VIC_with_endangered_purpose.csv')
sa_qld_df = pd.read_csv('/content/Iteration3_SA_QLD_with_endangered_purpose.csv')
other_states_df = pd.read_csv('/content/Iteration3_Species_otherstates_with_purpose.csv')


In [3]:
merged_df = pd.concat([vic_df, sa_qld_df, other_states_df], ignore_index=True)

In [5]:
grouped_df = merged_df.groupby(
    ['species_common_name', 'species_scientific_name', 'state', 'month', 'is_endangered', 'migration_purpose'],
    as_index=False
)['observation_count'].sum().rename(columns={
    'observation_count': 'total_observation_count',
    'migration_purpose': 'status'
})


In [ ]:
#grouped_df = grouped_df.sort_values(['species_common_name', 'state', 'month'])

In [6]:
grouped_df.head(5)
grouped_df.describe()

,total_observation_count
count,20747.000000
mean,54.670844
std,62.010087
min,1.000000
25%,7.000000
50%,13.000000
75%,108.000000
max,306.000000


In [23]:
grouped_df.to_csv('Iteration3_Grouped_Summary.csv', index=False)

**Modelling**

**Bird Attraction Predictor: Overview**


*Goal: Predict the likelihood of observing a particular bird species in a region (state + lat/lon) for a given month.*

In [9]:
grouped_df = pd.read_csv('/content/Iteration3_Grouped_Summary.csv')

# For every state and month, get the top 10 species by total_observation_count
top10_per_state_month = (
    grouped_df.sort_values(['state', 'month', 'total_observation_count', 'status'])
    .groupby(['state', 'month'])
    .head(10)
    .reset_index(drop=True)
)

In [10]:
output_path = "Top10_BirdSpecies_PerState_PerMonth.csv"
top10_per_state_month.to_csv(output_path, index=False)

output_path

'Top10_BirdSpecies_PerState_PerMonth.csv'

In [11]:
#Check
#  month is consistent (capitalize first letter)
top10_per_state_month['month'] = top10_per_state_month['month'].str.capitalize()

# List of all 12 months
all_months = ['January', 'February', 'March', 'April', 'May', 'June',
              'July', 'August', 'September', 'October', 'November', 'December']

# Cross-check per state
missing_months = {}

for state in top10_per_state_month['state'].unique():
    state_months = top10_per_state_month[top10_per_state_month['state'] == state]['month'].unique()
    missing = set(all_months) - set(state_months)
    if missing:
        missing_months[state] = sorted(list(missing))

# Show missing months if any
if missing_months:
    for state, months in missing_months.items():
        print(f"Missing months for {state}: {months}")
else:
    print("All states have all 12 months covered.")

All states have all 12 months covered.


In [12]:
# Month Standardization
top10_per_state_month['month'] = top10_per_state_month['month'].str.capitalize()

# Pivot table to count species per state per month
check_counts = top10_per_state_month.groupby(['state', 'month']).size().reset_index(name='species_count')

# Find where species_count < 10
incomplete_entries = check_counts[check_counts['species_count'] < 10]

if not incomplete_entries.empty:
    print("States with less than 10 species for some months:\n")
    print(incomplete_entries)
else:
    print("All states have top 10 bird species for every month.")


All states have top 10 bird species for every month.


In [15]:
top10_species = (
    top10_per_state_month.groupby('species_common_name')['total_observation_count']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .index.tolist()
)

print("Selected Top 10 Birds:", top10_species)

Selected Top 10 Birds: ['Grey Falcon', 'Yellow-breasted Boatbill', 'Cotton Pygmy-goose', 'Painted Honeyeater', 'Song Thrush', 'Australasian Bittern', 'Southern Scrub-robin', 'White-throated Needletail', 'Gentoo Penguin', 'Plains-wanderer']


In [22]:
df_grouped_summary = pd.read_csv('/content/Iteration3_Grouped_Summary.csv')

# Top 10 species per state and month
df_top10_per_state_month = (
    df_grouped_summary.sort_values(['state', 'month', 'total_observation_count'], ascending=[True, True, False])
    .groupby(['state', 'month'])
    .head(10)
    .reset_index(drop=True)
)

# Static Top 10 species across entire dataset (all months, all states)
list_top10_species = (
    df_top10_per_state_month.groupby('species_common_name')['total_observation_count']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .index.tolist()
)

# Filter dataset only for these species
df_filtered_top10_species = df_top10_per_state_month[df_top10_per_state_month['species_common_name'].isin(list_top10_species)]

# Select only dashboard-ready columns
df_dashboard_ready_top10 = df_filtered_top10_species[[
    'species_common_name',
    'state',
    'month',
    'total_observation_count',
    'is_endangered'
]]

# Sorting for dashboard clarity
df_dashboard_ready_top10 = df_dashboard_ready_top10.sort_values(['species_common_name', 'state', 'month'])

# Save for dashboard
df_dashboard_ready_top10.to_csv('Dashboard_Top10_Birds_Monthly_Migration.csv', index=False)


In [25]:
# Load full dataset with species, state, month, counts
df = pd.read_csv('Iteration3_Grouped_Summary.csv')

# Aggregate to get Top 10 birds by total observation count (all states, all months combined)
list_top10_birds_year = (
    df.groupby('species_common_name')['total_observation_count']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .index.tolist()
)

print("Top 10 Birds for the Year:", list_top10_birds_year)


Top 10 Birds for the Year: ['Australasian Bittern', 'Grey Falcon', 'Australian White Ibis', 'Plumed Egret', 'Cotton Pygmy-goose', 'Painted Honeyeater', 'Yellow-breasted Boatbill', 'Plains-wanderer', 'Pilotbird', 'Southern Scrub-robin']


**Getting Top 10 Birds**

In [29]:
df_top10_year = df[df['species_common_name'].isin(list_top10_birds_year)]

# Ensure key columns present for migration path visualization
df_top10_dashboard_ready = df_top10_year[[
    'species_common_name',
    'state',
    'month',
    'total_observation_count',
    'is_endangered',
    'status'
]]

# Export for dashboard
df_top10_dashboard_ready.to_csv('Dashboard_Top10_Birds_FullYear_Migration.csv', index=False)